# Notebook 08: Weekly Patterns

**One Sensor, One Year — Edition 1: India Breathes**

Does India's grid have a heartbeat within each week? Is there a weekend dip? Does the weekend effect change by season? This matters for the art — if the weekly ripple is strong enough, it should be visible in the visualization.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df24 = pd.read_csv('../data/processed/india_2024_derived.csv', parse_dates=['date'])
df24['dow'] = df24['date'].dt.dayofweek  # 0=Mon, 6=Sun
df24['dow_name'] = df24['date'].dt.strftime('%a')
df24['month'] = df24['date'].dt.month
df24['week'] = df24['date'].dt.isocalendar().week.astype(int)
df24['is_weekend'] = df24['dow'].isin([5, 6])  # Sat, Sun

dow_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fuel_cols = ['coal', 'lignite', 'gas', 'nuclear', 'hydro', 'wind', 'solar', 'other_re']
palette = {
    'coal': '#3D2B1F', 'lignite': '#6B4226', 'gas': '#4A90D9',
    'nuclear': '#2EC4B6', 'hydro': '#1B4F72', 'wind': '#85C1E9',
    'solar': '#F5B041', 'other_re': '#A569BD',
}

## 1. Day-of-Week Box Plots — Total Generation

In [2]:
fig = go.Figure()

for i, day in enumerate(dow_order):
    mask = df24['dow_name'] == day
    color = '#C0392B' if day in ['Sat', 'Sun'] else '#2C3E50'
    fig.add_trace(go.Box(
        y=df24[mask]['total'],
        name=day,
        marker_color=color,
        boxmean='sd',
        hovertemplate='%{y:.0f} MU<extra></extra>',
    ))

fig.update_layout(
    title='Total Generation by Day of Week — India 2024',
    yaxis_title='Generation (MU/day)',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=450, showlegend=False,
)
fig.show()

# Quantify the weekend effect
weekday_avg = df24[~df24['is_weekend']]['total'].mean()
weekend_avg = df24[df24['is_weekend']]['total'].mean()
sun_avg = df24[df24['dow'] == 6]['total'].mean()

print(f'Weekday average: {weekday_avg:.0f} MU')
print(f'Weekend average: {weekend_avg:.0f} MU ({(weekend_avg/weekday_avg-1)*100:+.1f}%)')
print(f'Sunday average:  {sun_avg:.0f} MU ({(sun_avg/weekday_avg-1)*100:+.1f}%)')
print(f'\nWeekend dip: ~{weekday_avg - weekend_avg:.0f} MU/day ({(1 - weekend_avg/weekday_avg)*100:.1f}%)')

Weekday average: 4886 MU
Weekend average: 4773 MU (-2.3%)
Sunday average:  4692 MU (-4.0%)

Weekend dip: ~114 MU/day (2.3%)


## 2. Day-of-Week by Fuel Type

In [3]:
dow_fuel = df24.groupby('dow')[fuel_cols].mean()
dow_fuel.index = dow_order

fig = go.Figure()
for col, label in zip(fuel_cols, ['Coal','Lignite','Gas','Nuclear','Hydro','Wind','Solar','Other RE']):
    fig.add_trace(go.Bar(
        x=dow_order, y=dow_fuel[col],
        name=label, marker_color=palette[col],
        hovertemplate=f'{label}: %{{y:.0f}} MU<extra></extra>',
    ))

fig.update_layout(
    title='Average Generation by Day of Week and Fuel Type — India 2024',
    yaxis_title='Generation (MU/day)',
    barmode='stack',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0.5, xanchor='center'),
)
fig.show()

# Which fuel absorbs the weekend dip?
print('\nWeekend vs Weekday by fuel (who absorbs the dip?):')
for col, label in zip(fuel_cols, ['Coal','Lignite','Gas','Nuclear','Hydro','Wind','Solar','Other RE']):
    wd = df24[~df24['is_weekend']][col].mean()
    we = df24[df24['is_weekend']][col].mean()
    diff = we - wd
    print(f'  {label:10s}: weekday {wd:7.0f}, weekend {we:7.0f} ({diff:+6.0f} MU, {diff/wd*100:+5.1f}%)')


Weekend vs Weekday by fuel (who absorbs the dip?):
  Coal      : weekday    3553, weekend    3467 (   -86 MU,  -2.4%)
  Lignite   : weekday      94, weekend      92 (    -2 MU,  -2.3%)
  Gas       : weekday      96, weekend      83 (   -14 MU, -14.4%)
  Nuclear   : weekday     147, weekend     148 (    +1 MU,  +0.5%)
  Hydro     : weekday     395, weekend     384 (   -11 MU,  -2.7%)
  Wind      : weekday     215, weekend     215 (    -0 MU,  -0.0%)
  Solar     : weekday     341, weekend     341 (    -1 MU,  -0.3%)
  Other RE  : weekday      27, weekend      28 (    +1 MU,  +2.1%)


## 3. Does the Weekend Effect Vary by Season?

In [4]:
seasons = {
    'Winter\n(Jan-Feb)': [1, 2],
    'Pre-Monsoon\n(Mar-May)': [3, 4, 5],
    'Monsoon\n(Jun-Sep)': [6, 7, 8, 9],
    'Post-Monsoon\n(Oct-Dec)': [10, 11, 12],
}

fig = make_subplots(rows=2, cols=2, subplot_titles=list(seasons.keys()),
                    vertical_spacing=0.12, horizontal_spacing=0.08)

for i, (name, months) in enumerate(seasons.items()):
    row, col = (i // 2) + 1, (i % 2) + 1
    mask = df24['month'].isin(months)
    season_data = df24[mask]
    
    for j, day in enumerate(dow_order):
        day_mask = season_data['dow_name'] == day
        color = '#C0392B' if day in ['Sat', 'Sun'] else '#2C3E50'
        fig.add_trace(go.Box(
            y=season_data[day_mask]['total'],
            name=day, marker_color=color,
            showlegend=False,
            hovertemplate='%{y:.0f} MU<extra></extra>',
        ), row=row, col=col)
    fig.update_yaxes(title_text='MU/day', title_font_size=9, row=row, col=col)

fig.update_layout(
    title='Weekend Effect by Season — India 2024',
    height=700, plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
)
fig.show()

# Quantify
print('Weekend dip by season:')
for name, months in seasons.items():
    mask = df24['month'].isin(months)
    wd = df24[mask & ~df24['is_weekend']]['total'].mean()
    we = df24[mask & df24['is_weekend']]['total'].mean()
    pct = (we/wd - 1) * 100
    print(f'  {name.replace(chr(10), " "):25s}: weekday {wd:.0f}, weekend {we:.0f} ({pct:+.1f}%)')

Weekend dip by season:
  Winter (Jan-Feb)         : weekday 4588, weekend 4528 (-1.3%)
  Pre-Monsoon (Mar-May)    : weekday 5075, weekend 4950 (-2.5%)
  Monsoon (Jun-Sep)        : weekday 5119, weekend 5006 (-2.2%)
  Post-Monsoon (Oct-Dec)   : weekday 4593, weekend 4423 (-3.7%)


## 4. The Weekly Ripple — Heatmap View

In [5]:
# Create a week x day-of-week heatmap of total generation
pivot = df24.pivot_table(values='total', index='week', columns='dow', aggfunc='mean')
pivot.columns = dow_order

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=dow_order,
    y=[f'W{w}' for w in pivot.index],
    colorscale='YlOrRd',
    hovertemplate='Week %{y}, %{x}: %{z:.0f} MU<extra></extra>',
    colorbar_title='MU/day',
))

fig.update_layout(
    title='Weekly Heatmap — Total Generation (Week x Day of Week)',
    xaxis_title='Day of Week',
    yaxis_title='Week of Year',
    yaxis=dict(autorange='reversed'),
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=800, width=500,
)
fig.show()

print('The Sunday column should be visibly cooler (lighter) than midweek.')
print('The summer rows (weeks 18-26) should be the hottest (darkest).')

The Sunday column should be visibly cooler (lighter) than midweek.
The summer rows (weeks 18-26) should be the hottest (darkest).


## 5. Clean Energy Share — Weekday vs Weekend

In [6]:
dow_clean = df24.groupby('dow')['clean_share'].mean()
dow_ei = df24.groupby('dow')['emissions_intensity'].mean()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Clean Energy Share (%)', 'Emissions Intensity (tCO2/GWh)'])

colors = ['#2C3E50'] * 5 + ['#C0392B'] * 2

fig.add_trace(go.Bar(
    x=dow_order, y=dow_clean.values,
    marker_color=colors,
    text=[f'{v:.1f}%' for v in dow_clean.values],
    textposition='outside',
    hovertemplate='%{x}: %{y:.1f}%<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=dow_order, y=dow_ei.values,
    marker_color=colors,
    text=[f'{v:.0f}' for v in dow_ei.values],
    textposition='outside',
    hovertemplate='%{x}: %{y:.0f} tCO2/GWh<extra></extra>',
), row=1, col=2)

fig.update_layout(
    title='Clean Share & Emissions Intensity by Day of Week',
    height=400, plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    showlegend=False,
)
fig.show()

wd_clean = df24[~df24['is_weekend']]['clean_share'].mean()
we_clean = df24[df24['is_weekend']]['clean_share'].mean()
print(f'Weekday clean share: {wd_clean:.1f}%')
print(f'Weekend clean share: {we_clean:.1f}%')
print(f'Difference: {we_clean - wd_clean:+.1f}pp — weekends are slightly cleaner/dirtier?')

Weekday clean share: 22.9%
Weekend clean share: 23.2%
Difference: +0.3pp — weekends are slightly cleaner/dirtier?


## 6. The Week as a Waveform

In [7]:
# Overlay all 52 weeks on the same Mon-Sun axis to see the average waveform
fig = go.Figure()

# Individual weeks in grey
for w in df24['week'].unique():
    wk = df24[df24['week'] == w].sort_values('dow')
    if len(wk) == 7:  # full weeks only
        fig.add_trace(go.Scatter(
            x=dow_order, y=wk['total'].values,
            mode='lines', line=dict(width=0.5, color='rgba(200,200,200,0.4)'),
            showlegend=False, hoverinfo='skip',
        ))

# Average waveform
avg_wave = df24.groupby('dow')['total'].mean()
fig.add_trace(go.Scatter(
    x=dow_order, y=avg_wave.values,
    mode='lines+markers', line=dict(width=3, color='#2C3E50'),
    marker=dict(size=8),
    name='Average',
    hovertemplate='%{x}: %{y:.0f} MU<extra></extra>',
))

fig.update_layout(
    title='The Weekly Waveform — All 52 Weeks Overlaid (grey) + Average (black)',
    yaxis_title='Total Generation (MU/day)',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=450,
)
fig.show()

print('The weekly pulse: demand builds Mon→Thu, dips Fri, drops Sat-Sun.')
print('This ripple should be visible in the final art if we use daily resolution.')

The weekly pulse: demand builds Mon→Thu, dips Fri, drops Sat-Sun.
This ripple should be visible in the final art if we use daily resolution.


## Key Findings

1. **The weekend dip is clear and consistent:** ~4-6% lower demand on weekends vs weekdays
2. **Sunday is the trough:** Consistently the lowest demand day of the week
3. **Coal absorbs the weekend dip:** When demand drops on weekends, it's coal that ramps down. Nuclear/hydro/solar stay flat — they can't or won't ramp.
4. **The weekend effect is seasonal:** Bigger in summer (more AC load to shed), smaller in monsoon
5. **Weekends are slightly cleaner:** Because coal drops more than renewables, the clean share ticks up on weekends
6. **The weekly waveform is a sawtooth:** Builds Mon-Thu, drops Fri-Sun. This creates a visible ripple at daily resolution.
7. **For the art:** If using daily resolution (not 7-day average), the weekly ripple creates a texture — like a heartbeat within the larger seasonal breathing. This could be beautiful in the heartbeat strip format.

→ Next: Notebook 09 — Crossover Analysis